In [2]:
# Step 1 - Initialize the Spark session for the Bronze ingestion layer
# This notebook reads the raw CSV source from S3, enriches it with ingestion
# metadata, and writes it into an Iceberg Bronze table managed by Nessie.

import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")

if src_path not in sys.path:
    sys.path.append(src_path)

from lakehouse.spark_session import get_spark
from lakehouse.settings import AWS_BUCKET


spark = get_spark("bronze-ingestion")

In [3]:
# Step 2 - Read the raw CSV sales dataset from the S3 raw zone
# The file is parsed with the proper CSV options to correctly
# handle quotes and commas.

sales_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(f"s3a://{AWS_BUCKET}/raw/sales/superstore_sales.csv")
)

sales_raw.show(5, truncate=False)
sales_raw.printSchema()

+--------------+----------+----------+--------------+-----------+-------------+-----------+-------------+------------+------------+-----------+-------+---------------+---------------+------------+------------------------------------------------------------------------+-------+--------+--------+--------+
|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name|Segment    |Country      |City        |State       |Postal Code|Region |Product ID     |Category       |Sub-Category|Product Name                                                            |Sales  |Quantity|Discount|Profit  |
+--------------+----------+----------+--------------+-----------+-------------+-----------+-------------+------------+------------+-----------+-------+---------------+---------------+------------+------------------------------------------------------------------------+-------+--------+--------+--------+
|CA-2019-103800|2019-01-03|2019-01-07|Standard Class|DP-13000   |Darren Powers|Consum

In [4]:
# Step 3 - Enrich the raw dataset with technical ingestion metadata
# These fields improve traceability and are typical of a Bronze layer.

from pyspark.sql.functions import current_timestamp, current_date, lit

sales_bronze_prepared = (
    sales_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("superstore_sales.csv"))
    .withColumn("source_system", lit("superstore_csv"))
    .withColumn("batch_id", lit("batch_001"))
)

sales_bronze_prepared.show(5, truncate=False)
sales_bronze_prepared.printSchema()

+--------------+----------+----------+--------------+-----------+-------------+-----------+-------------+------------+------------+-----------+-------+---------------+---------------+------------+------------------------------------------------------------------------+-------+--------+--------+--------+--------------+--------------------------+--------------------+--------------+---------+
|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name|Segment    |Country      |City        |State       |Postal Code|Region |Product ID     |Category       |Sub-Category|Product Name                                                            |Sales  |Quantity|Discount|Profit  |ingestion_date|ingestion_ts              |source_file         |source_system |batch_id |
+--------------+----------+----------+--------------+-----------+-------------+-----------+-------------+------------+------------+-----------+-------+---------------+---------------+------------+------------------

In [5]:
# Step 4 - Reset the Bronze sales table if needed
# This is useful during development to rebuild the Bronze layer from scratch.

spark.sql("DROP TABLE IF EXISTS nessie.bronze.sales")

DataFrame[]

In [6]:
# Step 5 - Write the raw sales dataset into the Bronze layer as an Iceberg table
# The table is stored in S3 and registered in the Nessie catalog.

from pyspark.sql.functions import col

(
    sales_bronze_prepared.writeTo("nessie.bronze.sales")
    .using("iceberg")
    .partitionedBy(col("ingestion_date"))
    .create()
)

In [ ]:
# Step 6 - Query the Bronze table to verify that the raw data was written correctly

spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").show()
spark.sql("SELECT * FROM nessie.bronze.sales LIMIT 10").show(truncate=False)

+--------+
|count(1)|
+--------+
|    9994|
+--------+

+--------------+----------+----------+--------------+-----------+-------------+-----------+-------------+------------+------------+-----------+-------+---------------+---------------+------------+------------------------------------------------------------------------+-------+--------+--------+--------+--------------+--------------------------+--------------------+--------------+---------+
|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name|Segment    |Country      |City        |State       |Postal Code|Region |Product ID     |Category       |Sub-Category|Product Name                                                            |Sales  |Quantity|Discount|Profit  |ingestion_date|ingestion_ts              |source_file         |source_system |batch_id |
+--------------+----------+----------+--------------+-----------+-------------+-----------+-------------+------------+------------+-----------+-------+-------

In [9]:
# Step 7 - Inspect the Iceberg metadata tables for the Bronze sales table
# The history table shows the snapshot lineage, and the snapshots table
# provides details about each version of the table.

spark.sql("SELECT * FROM nessie.bronze.sales.history").show(truncate=False)
spark.sql("SELECT * FROM nessie.bronze.sales.snapshots").show(truncate=False)

+-----------------------+-------------------+---------+-------------------+
|made_current_at        |snapshot_id        |parent_id|is_current_ancestor|
+-----------------------+-------------------+---------+-------------------+
|2026-03-12 00:35:52.731|1561604758626476291|NULL     |true               |
+-----------------------+-------------------+---------+-------------------+

+-----------------------+-------------------+---------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [10]:
# Step 8 - Simulate a new ingestion event by appending additional rows
# This creates a new Iceberg snapshot in the Bronze layer.

sales_bronze_prepared.limit(5).writeTo("nessie.bronze.sales").append()

In [11]:
# Step 9 - Query the metadata tables again to verify that a new snapshot
# was created after the append operation.

spark.sql("SELECT * FROM nessie.bronze.sales.history").show(truncate=False)
spark.sql("SELECT * FROM nessie.bronze.sales.snapshots").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-03-12 00:35:52.731|1561604758626476291|NULL               |true               |
|2026-03-12 00:37:14.81 |5918893253545800666|1561604758626476291|true               |
+-----------------------+-------------------+-------------------+-------------------+

+-----------------------+-------------------+-------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------